# Results 3 — Systematic colocalisation and effector-gene prediction

What supports each L2G gene assignment, and how that support changed over time.

The model-evaluation numbers of this subsection (average precision 0.81, area under the curve
0.95, recall 0.65, the false discovery rates of Supplementary Table 12, and the comparison
against the previous Open Targets model) need the L2G training set and the saved held-out
split, neither of which is available; see GAPS.md. The same applies to the loss-of-function
constraint enrichment (OR = 1.9) and to the secondary-signal analysis of Supplementary
Results 5.

Also writes the Extended Data Figure 5 Venn counts and the Extended Data Figure 6 temporal
table.

In [ ]:
import pandas as pd
from gentropy.common.session import Session
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
numbers = {}

In [ ]:
scored = (
    session.spark.read.parquet(paper.release("l2g_feature_matrix"))
    .filter(f.col("isProteinCoding") == 1)
    .select("studyLocusId", "geneId")
)
numbers["R3.04"] = scored.count()
print("credible set to protein-coding gene pairs scored:", numbers["R3.04"])

## What supports each assignment

Over every gene prioritisation on a qualifying credible set, disease or measurement.

In [ ]:
assignments = (
    session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))
    .select("studyLocusId", "geneId", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS")
    .unionByName(
        session.spark.read.parquet(paper.derived("prioritised_genes_measurements")).select(
            "studyLocusId", "geneId", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS"
        )
    )
    .distinct()
    .toPandas()
)
total = len(assignments)
print("CS-gene prioritisations:", total)

numbers["R3.10"] = round(100 * assignments["eQTL_coloc"].mean(), 1)
numbers["R3.11"] = round(100 * assignments["pQTL_coloc"].mean(), 1)
numbers["R3.15"] = round(100 * assignments["distanceTSS"].mean(), 1)
numbers["R3.17"] = round(100 * assignments["VEP"].mean(), 1)

nearest = assignments[assignments["distanceTSS"] == 1]
unsupported = (nearest["VEP"] == 0) & (nearest["eQTL_coloc"] == 0) & (nearest["pQTL_coloc"] == 0)
numbers["R3.16"] = round(100 * unsupported.mean(), 1)
print({k: numbers[k] for k in ["R3.10", "R3.11", "R3.15", "R3.16", "R3.17"]})

## Extended Data Figure 5 — the four reasons a gene can be prioritised

In [ ]:
venn = (
    assignments.groupby(["eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS"])
    .size()
    .rename("assignments")
    .reset_index()
    .sort_values("assignments", ascending=False)
)
venn.to_csv(paper.derived("extended_figure_5_venn.csv"), index=False)
venn

## Extended Data Figure 6 — support over time

For each year, a gene counts as supported if any credible set published up to that year and
prioritising it carried a protein-altering variant or a molQTL colocalisation.

In [ ]:
genes = (
    session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))
    .select("geneId", "year", "score", "eQTL_coloc", "pQTL_coloc", "VEP")
    .toPandas()
)

records = []
for year in range(2006, 2025):
    subset = genes[genes["year"] <= year]
    if subset.empty:
        continue
    per_gene = subset.groupby("geneId").agg(
        maxScore=("score", "max"),
        pav=("VEP", "max"),
        eqtl=("eQTL_coloc", "max"),
        pqtl=("pQTL_coloc", "max"),
    )
    coloc = (per_gene["eqtl"] == 1) | (per_gene["pqtl"] == 1)
    pav = per_gene["pav"] == 1
    records.append(
        {
            "year": year,
            "genes": len(per_gene),
            "meanMaxL2G": float(per_gene["maxScore"].mean()),
            "seMaxL2G": float(per_gene["maxScore"].sem()),
            "pavOnly": int((pav & ~coloc).sum()),
            "pavAndColoc": int((pav & coloc).sum()),
            "colocOnly": int((~pav & coloc).sum()),
            "neither": int((~pav & ~coloc).sum()),
        }
    )

temporal = pd.DataFrame(records)
temporal["pctNeither"] = 100 * temporal["neither"] / temporal["genes"]
temporal.to_csv(paper.derived("extended_figure_6_temporal.csv"), index=False)

by_year = temporal.set_index("year")["pctNeither"]
numbers["R3.08"] = round(float(by_year.loc[2015]), 0)
numbers["R3.09"] = round(float(by_year.loc[2024]), 0)
print({k: numbers[k] for k in ["R3.08", "R3.09"]})
temporal[["year", "genes", "meanMaxL2G", "pctNeither"]].round(3).tail(12)

## MST1, a non-nearest prioritisation

The inflammatory bowel disease credible set at 3:49676792:T/C, where L2G nominates the third
closest protein-coding gene.

In [ ]:
mst1 = (
    session.spark.read.parquet(paper.derived("prioritised_genes_annotated"))
    .filter(f.col("variantId") == "3_49676792_T_C")
    .select("variantId", "studyId", "geneId", "score", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS")
    .toPandas()
)
mst1_rows = mst1[mst1["geneId"] == "ENSG00000173531"]  # MST1
if len(mst1_rows):
    numbers["R3.18"] = round(float(mst1_rows["score"].max()), 2)
print("MST1 L2G score:", numbers.get("R3.18"))
mst1.sort_values("score", ascending=False).head(10)

## Numbers

In [ ]:
print(paper.save_results("colocalisation_l2g", numbers))
pd.Series(numbers).to_frame("computed")